# conditional-hparam-branch — ex2: conditional Dropout submodule — only register nn.Dropout when p > 0

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `conditional-hparam-branch`. Running the final beacon cell reports progress against the `PyTorch: Conditional hparam branch` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Conditional hparam branch` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`conditional-hparam-branch`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "conditional-hparam-branch"
DD_SUBTOPIC = "PyTorch: Conditional hparam branch"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Conditional Dropout branch — `p > 0` builds a real module, `p == 0` skips

Ex1 gated `bias=True/False` on `nn.Linear`. The deepening move handles the same conditional pattern for a regularization submodule: when `dropout > 0`, build an `nn.Dropout(p=dropout)` and apply it; when `dropout == 0`, skip the module entirely (don't even register it).

```python
class Block(nn.Module):
    def __init__(self, d_in, d_out, dropout: float):
        super().__init__()
        self.linear = nn.Linear(d_in, d_out)
        if dropout > 0:
            self.drop = nn.Dropout(p=dropout)
        # else: no self.drop attribute at all

    def forward(self, x):
        x = self.linear(x)
        if hasattr(self, 'drop'):
            x = self.drop(x)
        return x
```

**Why `hasattr` over `if self.drop is not None`.** Setting `self.drop = None` works in plain Python but registers `None` against the `nn.Module` child machinery in some versions and surprises `named_modules()`. Either skip the attribute entirely (cleanest) or use `nn.Identity()` as a no-op placeholder — both keep the forward path branch-free at the cost of one extra module slot.

**Same `nn.Linear` either way.** Dropout has zero parameters, so `len(list(model.parameters()))` is identical regardless of branch — the only difference is whether `model.modules()` includes a Dropout.

### Exercise 2 — conditional Dropout submodule — only register nn.Dropout when p > 0

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the conditional-submodule pattern: register `nn.Dropout(p)` as `self.drop` only when `p > 0`, and gate the forward call on `hasattr(self, 'drop')` so the `p == 0` path skips the module entirely (no `nn.Identity` placeholder).
> Keywords: dropout, conditional, submodule, hparam
> ```

**KCs targeted:** `conditional-submodule-registration`, `hasattr-gated-forward`

Implement `ex2_make_block(d_in, d_out, dropout)` — returns a `nn.Module` subclass instance.

Behavior:

1. The class (define it inside the helper or at module scope — your choice) is `nn.Module`-based.
2. In `__init__`:
   - Call `super().__init__()`.
   - Register `self.linear = nn.Linear(d_in, d_out)`.
   - If `dropout > 0`: register `self.drop = nn.Dropout(p=dropout)`.
   - If `dropout == 0`: do NOT register `self.drop` at all (no `nn.Identity`, no `self.drop = None`).
3. In `forward(x)`:
   - `x = self.linear(x)`.
   - If `hasattr(self, 'drop')`: `x = self.drop(x)`.
   - Return `x`.

Constraints:
- `dropout` is a float in `[0.0, 1.0)`.
- For `dropout == 0`, `dict(self.named_modules())` must NOT contain a `'drop'` key.
- For `dropout > 0`, it MUST contain a `'drop'` key whose value is an `nn.Dropout` instance.
- The Linear's parameter count must be unchanged regardless of the dropout branch.

In [ ]:
def ex2_make_block(d_in: int, d_out: int, dropout: float):
    """Build a Linear+optional-Dropout block. Dropout registered only when p > 0."""
    raise NotImplementedError()


def _test_ex2():
    import torch.nn as nn

    # === dropout == 0: no drop submodule at all ===
    block = ex2_make_block(4, 6, dropout=0.0)
    assert isinstance(block, nn.Module)
    named = dict(block.named_modules())
    assert 'drop' not in named, f'p=0 must NOT register drop; got modules {list(named.keys())}'
    assert 'linear' in named, f'linear must be registered; got modules {list(named.keys())}'
    assert not hasattr(block, 'drop'), 'with p=0, hasattr(block, drop) must be False'

    # === dropout > 0: drop submodule IS registered ===
    block = ex2_make_block(4, 6, dropout=0.5)
    assert hasattr(block, 'drop'), 'with p>0, block.drop must exist'
    assert isinstance(block.drop, nn.Dropout)
    assert abs(block.drop.p - 0.5) < 1e-9
    named = dict(block.named_modules())
    assert 'drop' in named and isinstance(named['drop'], nn.Dropout)

    # === Parameter count is the SAME either way (Dropout has 0 params) ===
    n_params_no_drop = sum(p.numel() for p in ex2_make_block(4, 6, 0.0).parameters())
    n_params_with_drop = sum(p.numel() for p in ex2_make_block(4, 6, 0.5).parameters())
    assert n_params_no_drop == n_params_with_drop, (
        f'Dropout has no params; counts must match. got {n_params_no_drop} vs {n_params_with_drop}'
    )
    # 4*6 + 6 = 30
    assert n_params_no_drop == 30

    # === Forward works in both branches (eval mode, no randomness) ===
    block = ex2_make_block(4, 6, dropout=0.0)
    block.eval()
    x = t.randn(3, 4)
    y = block(x)
    assert y.shape == (3, 6), f'forward output shape wrong: {y.shape}'

    block = ex2_make_block(4, 6, dropout=0.5)
    block.eval()  # dropout in eval mode is a no-op — deterministic
    y = block(x)
    assert y.shape == (3, 6)
    # In eval mode, dropout=0.5 should give the same output as dropout=0.0
    # (when both blocks have the same Linear weights — they won't here
    # because of separate random init, so just check the shape).

    # === Train mode: dropout > 0 actually drops in train() ===
    t.manual_seed(42)
    block = ex2_make_block(4, 6, dropout=0.9)
    block.train()
    x = t.ones(100, 4)
    y = block(x)
    n_zeros = (y == 0).sum().item()
    assert n_zeros > 0, f'with p=0.9 in train mode, some outputs should be zeroed; got n_zeros={n_zeros}'

    # === Train mode: dropout == 0 has zero zeros (with non-zero linear bias) ===
    block = ex2_make_block(4, 6, dropout=0.0)
    block.train()
    # Set the linear weights to all-1 and bias to all-1 so the output is deterministic non-zero.
    with t.no_grad():
        block.linear.weight.fill_(1.0)
        block.linear.bias.fill_(1.0)
    y = block(t.ones(10, 4))
    # Each output element should be 4*1 + 1 = 5.
    assert (y == 5.0).all(), f'p=0 must not zero anything; got y={y}'

    # === Linear submodule unchanged across branches ===
    for p in [0.0, 0.1, 0.5, 0.9]:
        block = ex2_make_block(8, 16, dropout=p)
        assert isinstance(block.linear, nn.Linear)
        assert block.linear.in_features == 8 and block.linear.out_features == 16
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
import torch.nn as nn

class _Block(nn.Module):
    def __init__(self, d_in, d_out, dropout):
        super().__init__()
        self.linear = nn.Linear(d_in, d_out)
        if dropout > 0:
            self.drop = nn.Dropout(p=dropout)
        # else: do nothing — no attribute set at all.

    def forward(self, x):
        x = self.linear(x)
        if hasattr(self, 'drop'):
            x = self.drop(x)
        return x

def ex2_make_block(d_in, d_out, dropout):
    return _Block(d_in, d_out, dropout)
```

**`if dropout > 0` is the registration gate.** `nn.Module.__setattr__` auto-registers any `nn.Module` value into `self._modules`. If you never assign `self.drop`, no registration happens — `named_modules()` doesn't see it, `state_dict()` doesn't have an entry, and `forward()` skips it via `hasattr`.

**Why not `self.drop = nn.Identity()`.** `nn.Identity()` ALSO registers (as a child module). Functionally equivalent at forward-time, but `named_modules()` then includes an extra entry that downstream introspection code (e.g. logging hooks, hardware compilers) has to learn to ignore. The drill picks the no-registration form as the strictest conditional pattern.

**Parameter count is invariant.** `nn.Dropout` has zero `Parameter`s — only a `p` config attribute. Whether you include it or not, `model.parameters()` returns the same list. This is what makes the conditional safe to toggle mid-sweep.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()